In [50]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, when, isnull

# %%
# Configurar SparkSession
spark = SparkSession.builder \
    .appName("SECOP_FeatureEngineering") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")

Spark Version: 3.5.0
Spark Master: spark://spark-master:7077


In [51]:
# Cargar datos
df = spark.read.parquet("/opt/spark-data/processed/secop_eda.parquet")
print(f"Registros cargados: {df.count():,}")

Registros cargados: 100,000


In [52]:
# Explorar columnas disponibles
print("Columnas disponibles:")
for col_name in df.columns:
    print(f"  - {col_name}")

Columnas disponibles:
  - referencia_del_contrato
  - nit_entidad
  - nombre_entidad
  - departamento
  - ciudad
  - tipo_de_contrato
  - valor_del_contrato
  - fecha_de_firma
  - duraci_n_del_contrato
  - proveedor_adjudicado
  - estado_contrato
  - year
  - month
  - valor_del_contrato_num
  - fecha_de_firma_parsed
  - anio
  - mes


In [53]:
# ## RETO 1: Selección de Features
# Seleccionar features para el modelo

# Variables categóricas

categorical_cols = ["tipo_de_contrato", "departamento", "ciudad"]
 
# Variables numéricas

numeric_cols = ["anio", "mes"]
 
# Label (no entra al modelo)
label_col = "valor_del_contrato_num"
 
# Verificar qué columnas existen
available_cat = [c for c in categorical_cols if c in df.columns]
available_num = [c for c in numeric_cols if c in df.columns]
 
print(f"Categóricas disponibles: {available_cat}")
print(f"Numéricas disponibles: {available_num}")
print(f"Label: {label_col}")

Categóricas disponibles: ['tipo_de_contrato', 'departamento', 'ciudad']
Numéricas disponibles: ['anio', 'mes']
Label: valor_del_contrato_num


Para la predicción del *valor del contrato*, se seleccionaron variables que capturan tanto
características estructurales del contrato como su contexto geográfico y temporal.

### Variables categóricas

- *tipo_de_contrato*:  
  Es una variable clave, ya que distintos tipos de contrato implican naturalmente  diferencias significativas en los montos adjudicados.

- *departamento*:  
  Permite capturar diferencias regionales en costos, presupuestos institucionales
  y niveles de inversión pública entre distintas zonas del país.

- *ciudad*:  
  Aporta un nivel de granularidad mayor que el departamento, reflejando variaciones locales
  en precios, demanda de servicios y capacidad económica.

  
### Variables numéricas

- *anio*:  
  Corresponde al año en que se firmó el contrato.  
  Es fundamental porque el valor de los contratos está altamente influenciado por
  factores macroeconómicos como inflación, cambios presupuestales y políticas públicas,
  que varían significativamente de un año a otro.

- *mes*:  
  Captura posibles patrones estacionales en la contratación pública,
  como picos de ejecución presupuestal a final de año.

### Conclusión

La combinación de variables categóricas (contexto y tipo de contrato) y numéricas (componente temporal)
permite construir un modelo más robusto, capaz de capturar tanto patrones estructurales
como dinámicos en el valor de los contratos.


In [54]:
label_col = "valor_del_contrato_num"

# Asegurar tipos numéricos 
df_cast = df \
    .withColumn("month", col("month").cast("int")) \
    .withColumn(label_col, col(label_col).cast("double"))

# Limpiar datos: eliminar nulos en features y label

df_clean = df.dropna(subset=available_cat + available_num + [label_col])
print(f"Registros después de limpiar nulos: {df_clean.count():,}") 

Registros después de limpiar nulos: 100,000


*Estrategia seleccionada: Opción A – Eliminar filas con nulos (dropna)*

Decidimos eliminar los registros con valores nulos en las variables seleccionadas 
(categóricas, numéricas y la variable objetivo) utilizando dropna.

Por que el dataset cuenta con un volumen suficientemente grande de datos (más de 100.000 registros), por lo que eliminar filas con valores nulos no afecta significativamente la representatividad del conjunto de datos. Al tratarse de un problema de regresión, es crítico no introducir ruido artificial en la variable objetivo (valor_del_contrato_num), lo cual podría ocurrir si se imputan valores de forma incorrecta. Esta estrategia garantiza que el modelo se entrene únicamente con datos completos,reales y consistentes, simplificando el pipeline y evitando sesgos derivados de imputaciones.

Por estas razones, se optó por una estrategia de limpieza conservadora basada en la eliminación de registros incompletos.

In [55]:
# Convierte strings a índices numéricos
indexers = [
    StringIndexer(inputCol=col, outputCol=col + "_idx", handleInvalid="keep")
    for col in available_cat
]

In [56]:
# ## PASO 1: StringIndexer para Variables Categóricas
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in available_cat
]

print("StringIndexers:")
for idx in indexers:
    print(" -", idx.getInputCol(), "->", idx.getOutputCol())


StringIndexers:
 - tipo_de_contrato -> tipo_de_contrato_idx
 - departamento -> departamento_idx
 - ciudad -> ciudad_idx


In [57]:
# PASO 2: OneHotEncoder para generar variables dummy
from pyspark.ml.feature import OneHotEncoder

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec")
    for c in available_cat
]

print("OneHotEncoders:")
for enc in encoders:
    print(" -", enc.getInputCol(), "->", enc.getOutputCol())

OneHotEncoders:
 - tipo_de_contrato_idx -> tipo_de_contrato_vec
 - departamento_idx -> departamento_vec
 - ciudad_idx -> ciudad_vec


In [58]:
# RETO 3: VectorAssembler para combinar todas las features
# Combinamos: features numéricas + features categóricas codificadas
feature_cols = []
feature_cols += available_num
feature_cols += [f"{c}_vec" for c in available_cat]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")


print("VectorAssembler inputCols:", feature_cols)

VectorAssembler inputCols: ['anio', 'mes', 'tipo_de_contrato_vec', 'departamento_vec', 'ciudad_vec']


*Pregunta de reflexión*: ¿Por qué necesitamos combinar features numéricas y categóricas
codificadas en un solo vector?

En Spark ML, todos los algoritmos de Machine Learning esperan recibir las variables de entrada en una única columna de tipo vector, generalmente llamada features.

Por esta razón, es necesario combinar tanto las variables numéricas originales como las variables categóricas previamente codificadas (OneHotEncoding) en un solo vector mediante VectorAssembler.

El VectorAssembler transforma múltiples columnas heterogéneas en una representación
numérica unificada, que puede ser procesada eficientemente por los algoritmos de Spark ML. Esto permite que el modelo aprenda patrones combinando información de diferentes tipos de variables (temporales, geográficas y categóricas) de forma conjunta.

En resumen, el VectorAssembler es un paso obligatorio para convertir un dataset tabular
en una representación matemática compatible con los modelos de Machine Learning.


In [59]:
# RETO 4: Construir Pipeline
# Pipeline = secuencia de transformaciones
pipeline_stages = []
pipeline_stages += indexers
pipeline_stages += encoders
pipeline_stages += [assembler]

pipeline = Pipeline(stages=pipeline_stages)

print("Pipeline stages:", [type(s).__name__ for s in pipeline_stages])

Pipeline stages: ['StringIndexer', 'StringIndexer', 'StringIndexer', 'OneHotEncoder', 'OneHotEncoder', 'OneHotEncoder', 'VectorAssembler']


*Pregunta*: ¿Cuál es el orden correcto de los stages?

### Respuesta:

El orden correcto de los stages en el Pipeline es:

*StringIndexer → OneHotEncoder → VectorAssembler*

Este orden es obligatorio debido a las dependencias entre los transformadores:

1. *StringIndexer*  
   Convierte las variables categóricas de texto a índices numéricos.  
   Es el primer paso porque los modelos no pueden trabajar directamente con strings.

2. *OneHotEncoder*  
   Recibe como entrada los índices generados por el StringIndexer y los transforma
   en vectores binarios (one-hot).  
   No puede ejecutarse antes del indexado.

3. *VectorAssembler*  
   Combina todas las variables (numéricas y categóricas codificadas) en un solo vector
   de features.  
   Debe ser el último paso porque necesita que todas las transformaciones previas
   ya estén listas.


In [60]:
# PASO 5: Entrenar el pipeline (fit)
# Nota: StringIndexer y OneHotEncoder necesitan "aprender" del dataset
print("Entrenando pipeline...")
pipeline_model = pipeline.fit(df_clean)
print("✓ Pipeline entrenado")

df_transformed = pipeline_model.transform(df_clean)

# Spark ML espera columna 'label'
df_model = df_transformed.withColumnRenamed(label_col, "label")

df_model.select("label", "features_raw").show(5, truncate=False)

sample_features = df_model.select("features_raw").first()[0]
print("Dimensión final features_raw:", len(sample_features))
print("Registros finales:", df_model.count())

Entrenando pipeline...


✓ Pipeline entrenado
+------------+--------------------------------------------+
|label       |features_raw                                |
+------------+--------------------------------------------+
|2.067E7     |(573,[0,1,2,31,66],[2025.0,1.0,1.0,1.0,1.0])|
|3.0848588E7 |(573,[0,1,2,19,54],[2025.0,1.0,1.0,1.0,1.0])|
|9.5E7       |(573,[0,1,2,18,53],[2025.0,1.0,1.0,1.0,1.0])|
|1.06659819E8|(573,[0,1,2,18,53],[2025.0,1.0,1.0,1.0,1.0])|
|7200000.0   |(573,[0,1,2,25,56],[2025.0,1.0,1.0,1.0,1.0])|
+------------+--------------------------------------------+
only showing top 5 rows

Dimensión final features_raw: 573
Registros finales: 100000


In [61]:
# %%
# Verificar el resultado
print("\nEsquema de features_raw:")
df_transformed.select("features_raw").printSchema()


Esquema de features_raw:
root
 |-- features_raw: vector (nullable = true)



In [62]:
# Ver dimensión del vector de features
sample_features = df_transformed.select("features_raw").first()[0]
print(f"Dimensión del vector de features: {len(sample_features)}")

Dimensión del vector de features: 573


In [63]:
# Ver dimensión del vector de features
sample_features = df_transformed.select("features_raw").first()[0]
print(f"Dimensión del vector de features: {len(sample_features)}")

# %%
# Mostrar ejemplo de transformación
df_transformed.select(
    available_cat[0] if available_cat else "id",
    available_cat[0] + "_idx" if available_cat else "id",
    available_cat[0] + "_vec" if available_cat else "id",
    "features_raw"
).show(5, truncate=True)

Dimensión del vector de features: 573
+--------------------+--------------------+--------------------+--------------------+
|    tipo_de_contrato|tipo_de_contrato_idx|tipo_de_contrato_vec|        features_raw|
+--------------------+--------------------+--------------------+--------------------+
|Prestación de ser...|                 0.0|      (16,[0],[1.0])|(573,[0,1,2,31,66...|
|Prestación de ser...|                 0.0|      (16,[0],[1.0])|(573,[0,1,2,19,54...|
|Prestación de ser...|                 0.0|      (16,[0],[1.0])|(573,[0,1,2,18,53...|
|Prestación de ser...|                 0.0|      (16,[0],[1.0])|(573,[0,1,2,18,53...|
|Prestación de ser...|                 0.0|      (16,[0],[1.0])|(573,[0,1,2,25,56...|
+--------------------+--------------------+--------------------+--------------------+
only showing top 5 rows



In [64]:
# %%
# Guardar pipeline entrenado
pipeline_path = "/opt/spark-data/processed/feature_pipeline"
pipeline_model.write().overwrite().save(pipeline_path)
print(f"✓ Pipeline guardado en: {pipeline_path}")

# %%
# Guardar dataset transformado
output_path = "/opt/spark-data/processed/secop_features.parquet"
df_model.write.mode("overwrite").parquet(output_path)
print(f"Dataset transformado guardado en: {output_path}")

✓ Pipeline guardado en: /opt/spark-data/processed/feature_pipeline


Dataset transformado guardado en: /opt/spark-data/processed/secop_features.parquet


## Preguntas de Reflexión

### 1. ¿Por qué usamos Pipeline en lugar de aplicar transformaciones individuales?

Porque el Pipeline permite *encadenar todas las transformaciones en un solo flujo reproducible*.  
Esto garantiza que los mismos pasos de preprocesamiento se apliquen de forma consistente
tanto en entrenamiento como en predicción, evitando errores humanos y asegurando
trazabilidad y mantenibilidad del proceso.
Además, facilita la validación, el versionado y la automatización en entornos productivos.


### 2. ¿Qué pasaría si aplicamos OneHotEncoder antes de StringIndexer?

No funcionaría correctamente, ya que *OneHotEncoder solo acepta valores numéricos*.
Las variables categóricas primero deben ser transformadas a índices numéricos mediante
StringIndexer.  
Si se invierte el orden, el pipeline generará un error porque intentaría codificar strings
directamente.


### 3. ¿Cuándo usarías StandardScaler en el pipeline?

Se utiliza cuando las variables numéricas tienen *escalas muy diferentes*
(por ejemplo, edad vs ingresos), especialmente en modelos sensibles a la magnitud
de las features como:

- Regresión lineal
- Regresión logística
- K-Means
- Redes neuronales

StandardScaler ayuda a que todas las variables tengan media 0 y desviación estándar 1,
mejorando la estabilidad y el rendimiento del modelo.


### 4. ¿Qué ventaja tiene guardar el pipeline_model en lugar del DataFrame transformado?

Guardar el pipeline_model permite *reutilizar exactamente las mismas transformaciones*
sobre nuevos datos sin necesidad de reentrenar todo el proceso.

Esto es clave en producción, ya que garantiza:
- Consistencia entre entrenamiento y predicción
- Reproducibilidad del modelo
- Facilidad de despliegue (solo se carga el pipeline y se transforma cualquier dataset nuevo)

Mientras que el DataFrame transformado solo sirve para ese conjunto específico de datos,
el pipeline es reutilizable y escalable.


In [65]:
# %%
print("\n" + "="*60)
print("RESUMEN FEATURE ENGINEERING")
print("="*60)
print(f"✓ Variables categóricas procesadas: {len(available_cat)}")
print(f"✓ Variables numéricas: {len(available_num)}")
print(f"✓ Dimensión final del vector: {len(sample_features)}")
print(f"✓ Pipeline guardado y listo para usar")
print("="*60)


RESUMEN FEATURE ENGINEERING
✓ Variables categóricas procesadas: 3
✓ Variables numéricas: 2
✓ Dimensión final del vector: 573
✓ Pipeline guardado y listo para usar


In [66]:
# RETO BONUS 1: ¿Cuántas features se generaron?
# ============================================================

print("\nRETO BONUS 1 - Conteo de categorías:")

total_cat_features = 0

for cat_col in available_cat:
    num_categorias = df_clean.select(cat_col).distinct().count()
    print(f"{cat_col}: {num_categorias} categorías únicas")
    total_cat_features += num_categorias

total_features_manual = len(available_num) + total_cat_features

print("\nCálculo manual:")
print("Features numéricas:", len(available_num))
print("Features categóricas (one-hot):", total_cat_features)
print("TOTAL FEATURES ESPERADAS:", total_features_manual)

print("\nVerificación real:")
print("TOTAL FEATURES REALES:", len(sample_features))



RETO BONUS 1 - Conteo de categorías:
tipo_de_contrato: 16 categorías únicas
departamento: 34 categorías únicas
ciudad: 521 categorías únicas

Cálculo manual:
Features numéricas: 2
Features categóricas (one-hot): 571
TOTAL FEATURES ESPERADAS: 573

Verificación real:
TOTAL FEATURES REALES: 573


## RETO BONUS 1: ¿Cuántas features se generaron?

### Cálculo teórico

En este experimento se utilizan:

- *2 variables numéricas*
- *3 variables categóricas*:
  - tipo_de_contrato: 16 categorías únicas  
  - departamento: 34 categorías únicas  
  - ciudad: 521 categorías únicas  

Cuando se aplica *OneHotEncoding*, cada categoría se convierte en una feature binaria (0/1).

Por lo tanto, el número total de features categóricas es:

16 + 34 + 521 = *571 features*

Sumando las variables numéricas:

571 (categóricas) + 2 (numéricas) = *573 features totales*

### Verificación con el modelo

El tamaño real del vector de features generado por Spark es:

```python
len(sample_features) = 573

In [67]:
# ============================================================
# RETO BONUS 2: Feature Importance Manual (Varianza)
# ============================================================

import pandas as pd
import numpy as np

print("\nRETO BONUS 2 - Varianza de features")

# Tomar muestra
sample_df = df_model.select("features_raw") \
    .sample(0.05) \
    .limit(1000) \
    .toPandas()

# Convertir a matriz numpy
features_matrix = np.array(
    [row['features_raw'].toArray() for _, row in sample_df.iterrows()]
)

# Calcular varianza
variances = np.var(features_matrix, axis=0)

# Top 5 features con mayor varianza
top_5_idx = np.argsort(variances)[-5:][::-1]

print("\nTop 5 features con mayor varianza:")
for idx in top_5_idx:
    print(f"Feature {idx}: varianza = {variances[idx]:.4f}")


RETO BONUS 2 - Varianza de features

Top 5 features con mayor varianza:
Feature 18: varianza = 0.2244
Feature 52: varianza = 0.1766
Feature 53: varianza = 0.1489
Feature 20: varianza = 0.1233
Feature 55: varianza = 0.1033


## RETO BONUS 2: Feature Importance Manual (Varianza)

### Metodología

1. Se tomó una *muestra aleatoria de 1000 registros* del dataset transformado.
2. Se convirtió el vector features_raw a una *matriz NumPy*.
3. Se calculó la *varianza de cada feature*.
4. Se identificaron las *5 features con mayor varianza*.

La varianza es una medida de dispersión:  
una feature con alta varianza aporta más información al modelo porque no es constante.

---

### Resultados

Top 5 features con mayor varianza:

| Feature | Varianza |
|--------:|----------|
| 18 | 0.2326 |
| 52 | 0.1885 |
| 53 | 0.1527 |
| 20 | 0.1190 |
| 19 | 0.0979 |

---

### Interpretación

Estas features son las que *más varían entre contratos*, por lo tanto:

Tienen *mayor poder informativo. Son las que probablemente **más influyen en el modelo* y corresponden principalmente a categorías con alta frecuencia o a variables numéricas.

En contraste, las features con varianza cercana a 0 aportan muy poca información y podrían eliminarse en un proceso de *feature selection* más avanzado.

---

### Conclusión

El análisis de varianza permite una forma simple pero efectiva de realizar una *feature importance manual*, identificando qué dimensiones del vector tienen mayor impacto potencial en la predicción del valor del contrato.


In [68]:

spark.stop()